# Classification de sentiment des avis TMDB (endpoint `/sentiment`)

**Perimetre** : Personne B (ML & NLP Engineer) -- voir `CLAUDE.md`.

**Objectif** : entrainer un modele de classification de sentiment
(negatif / neutre / positif) sur les avis TMDB stockes dans la table Gold
`avis`, pour alimenter l'endpoint `/sentiment`
(`api/routers/sentiment.py`, schema `SentimentScore`).

**Seuils a respecter** : F1 macro > 0.70 et accuracy > 0.72
(voir `CLAUDE.md`, section `/sentiment`).

**Changement methodologique important (a garder en tete tout au long du
notebook)** : ce notebook utilise desormais `avis.note_auteur` -- la vraie
note (0-10) laissee par l'auteur de l'avis TMDB lui-meme
(`author_details.rating`, ~94.6% de couverture) -- pour deriver le label de
sentiment. **Ce n'est pas l'approche originelle.** La toute premiere version
de ce notebook derivait le label de la note MovieLens laissee par un
utilisateur *synthetiquement* rattache a l'avis (tirage aleatoire parmi les
votants reels du film, pour respecter les contraintes de cle etrangere --
voir `pipeline/transform_silver.py::clean_avis`) : une supervision faible,
structurellement bruitee, puisque la note utilisee n'etait pas celle de
l'auteur reel de l'avis. Cette ancienne approche est **abandonnee** mais
conservee et rejouee dans la section 13 ("Comparaison historique") a titre
de trace honnete -- voir aussi `nlp/training/features.py` (docstring de
module) et `nlp/model_cards/distilbert_sentiment_v1.0_20260709.md` (model
card de l'ancienne version, conservee pour memoire).

**Structure du notebook** :
1. Chargement des donnees Gold (avis + notation)
2. EDA (couverture `note_auteur`, distribution des labels, longueur des avis)
3. Derivation du label (`note_auteur`) et repartition des classes
4. Split train/test stratifie (pas de fuite)
5. Baseline naive (classe majoritaire)
6. Baseline classique (TF-IDF + regression logistique)
7. Fine-tuning DistilBERT -- smoke test (~100 exemples)
8. Fine-tuning DistilBERT -- entrainement complet (charge le modele deja
   entraine par `python -m nlp.training.model`)
9. Comparaison des modeles
10. Matrice de confusion
11. F1 par classe
12. Exemples mal classes commentes
13. Comparaison historique avec l'ancien label (MovieLens synthetique) --
    trace honnete de l'approche abandonnee
14. Discussion des limites
15. Conclusion


In [ ]:
import sys
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    '''Remonte l'arborescence jusqu'a trouver le marqueur CLAUDE.md.

    Plus robuste que de supposer que le repertoire courant contient
    directement `nlp/` : fonctionne quel que soit le sous-dossier depuis
    lequel Jupyter/nbconvert a ete lance.
    '''
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "CLAUDE.md").exists():
            return candidate
    raise FileNotFoundError(
        "Impossible de localiser la racine du depot (marqueur CLAUDE.md introuvable)."
    )


REPO_ROOT = _find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

from nlp.training import features
from nlp.training import model as model_module

pd.set_option("display.max_colwidth", 200)
RANDOM_STATE = 42


## 1. Chargement des donnees Gold

Connexion directe a la base Gold (Supabase Postgres) via `DATABASE_URL`, memes utilitaires que `nlp/training/model.py` (pas de duplication de la logique de connexion). `notation` est charge mais n'est plus utilise pour deriver le label retenu -- il ne sert que pour la comparaison historique (section 13).


In [ ]:
avis_df, notation_df = model_module.load_gold_avis_data()
print(f"avis (Gold) : {len(avis_df)} lignes")
print(f"notation (Gold) : {len(notation_df)} lignes")
print(f"avis avec note_auteur renseignee : {avis_df['note_auteur'].notna().sum()} "
      f"({100 * avis_df['note_auteur'].notna().mean():.1f}%)")


## 2. EDA -- couverture, notes, longueur des avis

On construit d'abord le jeu de donnees labelise (derivation du label depuis
`note_auteur` + longueur du texte via `model_module.build_labeled_dataset`),
puis on regarde :
- la couverture des avis avec `note_auteur` renseignee,
- la distribution des notes/labels,
- la distribution de la longueur des avis (caracteres et mots).


In [ ]:
labeled_df = model_module.build_labeled_dataset(avis_df)
n_exclus = len(avis_df) - len(labeled_df)
print(f"Avis labelises (note_auteur non nulle) : {len(labeled_df)}")
print(f"Avis exclus (note_auteur manquante) : {n_exclus} sur {len(avis_df)}")
print()
print("Distribution des labels derives de note_auteur :")
print(features.label_distribution(labeled_df))
print()
print("Distribution des labels (%) :")
print((features.label_distribution(labeled_df) / len(labeled_df) * 100).round(1))


In [ ]:
print("Statistiques de longueur des avis (caracteres) :")
print(labeled_df["longueur_car"].describe())
print()
print("Statistiques de longueur des avis (mots) :")
print(labeled_df["longueur_mots"].describe())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(labeled_df["longueur_mots"], bins=40, color="steelblue")
axes[0].set_title("Longueur des avis (mots)")
axes[0].set_xlabel("nb mots")

axes[1].bar(
    features.label_distribution(labeled_df).index,
    features.label_distribution(labeled_df).values,
    color=["indianred", "gray", "seagreen"],
)
axes[1].set_title("Distribution des labels derives de note_auteur")
plt.tight_layout()
plt.show()


**Interpretation (EDA)** : *(section completee ci-dessous avec les chiffres
reels apres execution -- voir cellules precedentes pour les valeurs brutes).*


## 3. Derivation du label et repartition des classes

Regle de derivation (`nlp/training/features.py::derive_label_from_note_auteur`),
sur l'echelle 0-10 de `note_auteur` :
- `note_auteur <= 4` -> `negatif`
- `5 <= note_auteur <= 6` -> `neutre`
- `note_auteur >= 7` -> `positif`

Seuils choisis pour rester lisibles/explicables (convention usuelle des notes
sur 10) plutot que calibres finement sur la distribution empirique -- voir
la repartition de classes qui en resulte ci-dessous.

**Difference avec l'ancienne approche (abandonnee, section 13)** : `note_auteur`
est la vraie note laissee par l'auteur de l'avis TMDB lui-meme
(`author_details.rating`), pas celle d'un utilisateur MovieLens tiers
synthetiquement rattache. C'est une supervision fiable, pas une approximation
par distant supervision -- au prix d'exclure les ~5.4% d'avis sans
`note_auteur` plutot que de retomber sur l'ancien label bruite.


In [ ]:
print(labeled_df[["note_auteur", "label"]].groupby("label")["note_auteur"].describe())


## 4. Split train/test stratifie (pas de fuite)

Split stratifie sur le label (80/20), verification explicite qu'aucune paire
(user_id, film_id) n'apparait a la fois dans le train et le test.


In [ ]:
train_df, test_df = features.stratified_label_train_test_split(
    labeled_df, test_size=0.2, random_state=RANDOM_STATE
)

train_pairs = set(zip(train_df["user_id"], train_df["film_id"]))
test_pairs = set(zip(test_df["user_id"], test_df["film_id"]))
assert train_pairs.isdisjoint(test_pairs), "Fuite train/test detectee !"

print(f"Train : {len(train_df)}  Test : {len(test_df)}")
print("Pas de fuite train/test (paires user_id/film_id disjointes).")
print()
print("Repartition des labels (train) :")
print(features.label_distribution(train_df))
print("Repartition des labels (test) :")
print(features.label_distribution(test_df))


## 5. Baseline naive (classe majoritaire)

Predit systematiquement la classe majoritaire du train (`positif`). Sert de
reference plancher : tout modele doit faire mieux que cela pour justifier sa
complexite.


In [ ]:
baseline_metrics = model_module.evaluate_baseline_majority(train_df, test_df)
print(baseline_metrics)


## 6. Baseline classique (TF-IDF + regression logistique)

TF-IDF (uni + bi-grammes, `class_weight="balanced"` pour compenser le
desequilibre des classes) + regression logistique. Rapide a entrainer, sert
de reference "classique" avant d'investir dans le fine-tuning d'un
transformer.


In [ ]:
tfidf_pipeline, tfidf_metrics, tfidf_y_pred = model_module.train_tfidf_logreg(
    train_df, test_df
)
print(tfidf_metrics)
print()
print(
    features.compute_classification_report(test_df["label"].to_numpy(), tfidf_y_pred)
)


## 7. Fine-tuning DistilBERT -- smoke test (~100 exemples)

Conformement a `CLAUDE.md` ("tester sur ~100 exemples avant l'entrainement
complet"), on valide d'abord la plomberie complete (tokenisation, forme des
tenseurs, API Trainer, ponderation des classes) sur un tres petit
sous-echantillon et une seule epoque, avant d'investir le temps CPU
necessaire a l'entrainement complet (section 8). Les metriques ci-dessous ne
sont **pas representatives** (echantillon minuscule, 1 seule epoque) : seul
le bon fonctionnement du pipeline compte ici.


In [ ]:
small_train = train_df.sample(n=min(100, len(train_df)), random_state=RANDOM_STATE)
small_test = test_df.sample(n=min(50, len(test_df)), random_state=RANDOM_STATE)

_, _, smoke_metrics, _ = model_module.train_distilbert(
    small_train,
    small_test,
    output_dir=model_module.MODELS_DIR / "_smoke_test_checkpoints",
    num_train_epochs=1.0,
    per_device_batch_size=8,
)
print("Smoke test (non representatif) :", smoke_metrics)
print("Plomberie validee.")


## 8. Fine-tuning DistilBERT -- entrainement complet

**Choix assume** : le fine-tuning complet (2 epoques, batch=16) a ete
execute au prealable en arriere-plan via `python -m nlp.training.model`
(memes fonctions que ci-dessus, `random_state=42`, label derive de
`note_auteur`), pour un cout mesure d'environ 60-90 minutes CPU (pas de GPU
disponible sur cette machine). Ce notebook **charge le modele deja entraine**
plutot que de relancer un fine-tuning complet en ligne, afin de rester
executable en quelques minutes. Le modele charge est strictement celui qui
sera expose comme modele de production (meme artefact, memes poids),
version `v1.1` (voir `nlp/training/model.py::MODEL_VERSION`).


In [ ]:
import glob

candidate_dirs = sorted(
    d
    for d in glob.glob(str(model_module.MODELS_DIR / "distilbert_sentiment_v*"))
    if Path(d).is_dir()
)
assert candidate_dirs, (
    "Aucun modele distilbert sauvegarde trouve -- "
    "executez d'abord `python -m nlp.training.model`."
)
model_dir = candidate_dirs[-1]
print(f"Modele charge : {model_dir}")

bundle = model_module.load_model(model_dir)
print(f"Metriques enregistrees a l'entrainement : {bundle.metrics}")

distilbert_metrics, distilbert_y_pred = model_module.evaluate_saved_distilbert(
    bundle, test_df
)
print(f"Metriques recalculees sur le test set (ce notebook) : {distilbert_metrics}")


## 9. Comparaison des modeles


In [ ]:
comparison = pd.DataFrame(
    [
        {"modele": "baseline_majoritaire", **baseline_metrics},
        {"modele": "tfidf_logreg", **tfidf_metrics},
        {"modele": "distilbert", **distilbert_metrics},
    ]
)
baseline_f1 = baseline_metrics["macro_f1"]
comparison["amelioration_vs_baseline_naive_%"] = (
    (comparison["macro_f1"] - baseline_f1) / baseline_f1 * 100
).round(2)
print(comparison.to_string(index=False))

f1_ok = distilbert_metrics["macro_f1"] > model_module.F1_MACRO_THRESHOLD
acc_ok = distilbert_metrics["accuracy"] > model_module.ACCURACY_THRESHOLD
print()
print(f"Seuil F1 macro > {model_module.F1_MACRO_THRESHOLD} : {f1_ok}")
print(f"Seuil accuracy > {model_module.ACCURACY_THRESHOLD} : {acc_ok}")


## 10. Matrice de confusion (modele retenu)


In [ ]:
y_true = test_df["label"].to_numpy()
cm_df = features.compute_confusion_matrix(y_true, distilbert_y_pred)
print(cm_df)

fig, ax = plt.subplots(figsize=(5, 5))
ConfusionMatrixDisplay.from_predictions(
    y_true, distilbert_y_pred, labels=features.LABELS, ax=ax, colorbar=False
)
plt.title("Matrice de confusion -- DistilBERT (test set, label note_auteur)")
plt.tight_layout()
plt.show()


## 11. F1 par classe


In [ ]:
report_df = features.compute_classification_report(y_true, distilbert_y_pred)
print(report_df)

per_class = report_df.loc[features.LABELS, "f1-score"]
fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(per_class.index, per_class.values, color=["indianred", "gray", "seagreen"])
ax.axhline(model_module.F1_MACRO_THRESHOLD, color="black", linestyle="--", linewidth=1)
ax.set_title("F1 par classe -- DistilBERT (label note_auteur)")
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()


## 12. Exemples mal classes (commentes)


In [ ]:
misclassified = features.extract_misclassified(
    test_df.reset_index(drop=True), y_true, distilbert_y_pred, max_chars=280
)
print(
    f"{len(misclassified)} exemples mal classes sur {len(test_df)} "
    f"({100 * len(misclassified) / len(test_df):.1f}%)"
)
misclassified.head(10)


*(commentaire qualitatif complete ci-dessous apres inspection des exemples
ci-dessus, une fois le notebook execute avec les vraies predictions.)*


## 13. Comparaison historique avec l'ancien label (abandonne)

**Pourquoi cette section existe** : la toute premiere version de ce
notebook (voir `nlp/model_cards/distilbert_sentiment_v1.0_20260709.md`)
derivait le label de sentiment de la note MovieLens laissee par
l'utilisateur *synthetiquement* rattache a l'avis (tirage aleatoire parmi
les votants reels du film -- voir `pipeline/transform_silver.py::clean_avis`),
et non de la note du veritable auteur de l'avis. Cette approche a ete
**abandonnee** une fois `avis.note_auteur` disponible (couverture ~94.6%),
mais elle est rejouee ici -- via
`features.add_sentiment_labels_movielens_legacy` /
`model_module.build_labeled_dataset_legacy_movielens`, conservees dans le
code a cette seule fin -- pour documenter honnetement l'ecart entre les deux
approches plutot que de faire disparaitre la trace de ce choix.

Par souci de cout de calcul, seule la baseline TF-IDF+LogReg (rapide) est
reentrainee ici sur l'ancien label, a titre de comparaison ; le fine-tuning
DistilBERT complet n'est **pas** relance une seconde fois sur l'ancien label
(cout ~60-90 min CPU deja consenti une fois pour l'approche retenue) -- les
resultats DistilBERT de l'ancienne approche restent ceux documentes dans la
model card `v1.0`.


In [ ]:
legacy_labeled_df = model_module.build_labeled_dataset_legacy_movielens(
    avis_df, notation_df
)
print(f"Avis labelises (ancien label, note MovieLens) : {len(legacy_labeled_df)}")
print()
print("Distribution de l'ancien label (note MovieLens synthetique) :")
print(features.label_distribution(legacy_labeled_df))
print()
print("Distribution du nouveau label (note_auteur, rappel) :")
print(features.label_distribution(labeled_df))


In [ ]:
legacy_train_df, legacy_test_df = features.stratified_label_train_test_split(
    legacy_labeled_df, test_size=0.2, random_state=RANDOM_STATE
)
legacy_tfidf_pipeline, legacy_tfidf_metrics, legacy_tfidf_y_pred = (
    model_module.train_tfidf_logreg(legacy_train_df, legacy_test_df)
)

legacy_vs_new = pd.DataFrame(
    [
        {"approche": "ancienne (note MovieLens synthetique)", **legacy_tfidf_metrics},
        {"approche": "retenue (note_auteur)", **tfidf_metrics},
    ]
)
print("TF-IDF + LogReg, ancien label vs nouveau label (meme algorithme) :")
print(legacy_vs_new.to_string(index=False))


**Constat honnete** : *(a completer avec les chiffres reels ci-dessus apres
execution -- comparer la distribution des classes et le F1 macro TF-IDF
entre l'ancien label bruite et le nouveau label `note_auteur`)*. Le
changement de label modifie a la fois la difficulte du probleme (nouvelle
distribution de classes, potentiellement plus ou moins desequilibree) et la
fiabilite de la supervision (note reelle de l'auteur vs. note d'un tiers
synthetiquement rattache) -- les deux effets se melangent dans toute
comparaison brute de F1, d'ou l'importance de ne pas presenter un simple
delta de score comme une preuve d'amelioration du modele en soi.


## 14. Discussion des limites

*(section completee ci-dessous avec les chiffres reels apres execution)*


## 15. Conclusion

*(section completee ci-dessous avec les chiffres reels apres execution)*
